# Riverside Chatbot Function Breakdown

This notebook explains the chatbot in the simplest way possible.

How to read it:
- Each section shows one function or one small method.
- The explanation tells you what the code is doing in everyday words.
- The goal is not to sound clever. The goal is to make the code easy to understand.

Important note:
- These cells are mainly for learning.
- They show pieces of the real project code, one piece at a time.

## Notebook Setup

Run this setup cell first.

What it does:
- imports the basic Python tools used later
- creates the simple data shapes used in the notebook
- finds the project folder and the `faqs.json` file
- gives the later cells the names they need so they can run cleanly

In [2]:
from __future__ import annotations

import argparse
import json
import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field, replace
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Callable

import numpy as np


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "faqs.json").exists():
            return candidate
    raise FileNotFoundError("Could not find faqs.json from this notebook location.")


PROJECT_ROOT = find_project_root(Path.cwd())
DEFAULT_FAQS_FILE = PROJECT_ROOT / "faqs.json"


@dataclass(frozen=True)
class FAQ:
    id: int
    question: str
    answer: str


@dataclass(frozen=True)
class MatchCandidate:
    faq: FAQ
    score: float


@dataclass(frozen=True)
class MatchResult:
    faq: FAQ
    score: float
    margin: float
    matcher_name: str
    used_fallback: bool = False
    candidates: tuple[MatchCandidate, ...] = field(default_factory=tuple)


MODEL_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
DEFAULT_MIN_SCORE = 0.55
DEFAULT_MIN_MARGIN = 0.04
DEFAULT_LEXICAL_THRESHOLD = 0.3
DEFAULT_LEXICAL_MARGIN = 0.03
QUESTION_HINT_WEIGHT = 0.2
STOPWORDS = {
    "a", "all", "am", "an", "and", "any", "are", "at", "be", "bring",
    "can", "do", "for", "have", "how", "i", "in", "into", "is", "it",
    "me", "my", "of", "on", "or", "s", "store", "the", "there", "today",
    "to", "we", "what", "where", "you", "your",
}
TOKEN_ALIASES = {
    "cafe": {"coffee", "tea"},
    "coffee": {"cafe", "tea"},
    "friendly": {"accessible"},
    "open": {"hours"},
    "park": {"parking"},
    "parking": {"park"},
    "pickup": {"collect"},
    "site": {"website", "online"},
    "voucher": {"gift"},
    "vouchers": {"gift"},
    "website": {"online", "site"},
    "wheelchair": {"accessible"},
}

print(f"Setup complete. Project root: {PROJECT_ROOT}")

Setup complete. Project root: c:\Users\sossi\OneDrive\Desktop\agent builder workspace\Riverside Chatbot solution


## 1. `load_faqs`

This function opens the FAQ file and turns each item into a neat Python object.

Simple version:
- If you give it a file path, it uses that file.
- If you do not give it a file path, it uses the main `faqs.json` file.
- It reads the file.
- It turns each FAQ into an `FAQ` object.
- It gives back the full list.

In [3]:
def load_faqs(path: Path | None = None) -> list[FAQ]:
    """Load FAQ entries from disk."""

    faq_path = path or DEFAULT_FAQS_FILE
    with faq_path.open("r", encoding="utf-8") as handle:
        raw_items = json.load(handle)
    return [FAQ(**item) for item in raw_items]

## 2. `build_faq_document`

This function glues the FAQ question and answer together into one piece of text.

Why do that?
- Sometimes the question alone is not enough.
- The answer can also contain useful clues.
- So we search using both together.

In [4]:
def build_faq_document(faq: FAQ) -> str:
    """Combine the question and answer into one searchable document."""

    return f"Question: {faq.question}\nAnswer: {faq.answer}"

## 3. `_normalize_text`

This function cleans text into a more predictable form.

It does a few small fixes:
- turns everything into lower-case letters
- changes `wi-fi` into `wifi`
- changes `children's` into `children`
- changes `click-and-collect` into `click and collect`

This helps the chatbot treat similar phrases as the same idea.

In [12]:
def _normalize_text(text: str) -> str:
    lowered = text.lower()
    lowered = lowered.replace("wi-fi", "wifi")
    lowered = lowered.replace("children's", "children")
    lowered = lowered.replace("click-and-collect", "click and collect")
    return lowered

## 4. `_tokenize`

This function breaks a sentence into small useful words.

What it does:
- cleans the text first
- pulls out words and numbers
- removes very common words like `the`, `is`, and `you`
- adds a few helpful extra words when needed

Example idea:
- if the word is `site`, it can also add `website` and `online`

That gives the chatbot a better chance of spotting the real meaning.

In [6]:
def _tokenize(text: str) -> list[str]:
    base_tokens = re.findall(r"[a-z0-9]+", _normalize_text(text))
    expanded_tokens: list[str] = []
    for token in base_tokens:
        if token in STOPWORDS:
            continue
        expanded_tokens.append(token)
        expanded_tokens.extend(sorted(TOKEN_ALIASES.get(token, ())))
    return expanded_tokens

## 5. `_prepare_query`

This function gives the user's question a little extra help.

It does not change the meaning.
It just adds extra clue words when the user uses a short or casual phrase.

Examples:
- `where's the shop?` can get extra words like `location` and `address`
- `site` can get `website` and `online`
- `pick up an order` can get `click and collect`

This makes matching stronger without changing the user's intent.

In [7]:
def _prepare_query(query: str) -> str:
    lowered = _normalize_text(query)
    expansions: list[str] = []

    if "shop" in lowered and any(term in lowered for term in ("where", "address", "located")):
        expansions.append("location address located")
    if "site" in lowered:
        expansions.append("website online")
    if ("pick up" in lowered or "pickup" in lowered or "collect" in lowered) and (
        "online" in lowered or "order" in lowered
    ):
        expansions.append("click and collect")
    if "coffee" in lowered:
        expansions.append("cafe")
    if "wheelchair friendly" in lowered:
        expansions.append("wheelchair accessible")
    if "open" in lowered and "time" in lowered:
        expansions.append("opening hours")

    if not expansions:
        return query
    return f"{query} {' '.join(expansions)}"

## 6. `_cosine_similarity`

This function compares two lists of numbers and says how close they are.

Very simple meaning:
- bigger result means "more alike"
- smaller result means "less alike"

The chatbot uses this after the model turns text into number lists.

In [8]:
def _cosine_similarity(left: np.ndarray, right: np.ndarray) -> float:
    left_norm = float(np.linalg.norm(left))
    right_norm = float(np.linalg.norm(right))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    return float(np.dot(left, right) / (left_norm * right_norm))

## 7. `_lexical_similarity`

This function compares two pieces of text by looking at the words inside them.

It checks three things:
- how many important words from the question appear in the other text
- how focused that overlap is
- how similar the full text looks as a rough string match

Then it mixes those three parts into one score.

This is the backup matching method when the embedding model is not available.

In [9]:
def _lexical_similarity(query: str, text: str) -> float:
    query_tokens = set(_tokenize(_prepare_query(query)))
    if not query_tokens:
        return 0.0

    document_tokens = set(_tokenize(text))
    intersection = query_tokens & document_tokens
    if not intersection:
        return 0.0

    coverage = len(intersection) / len(query_tokens)
    precision = len(intersection) / len(document_tokens)
    fuzzy = SequenceMatcher(
        None,
        _normalize_text(query),
        _normalize_text(text),
    ).ratio()
    return (0.7 * coverage) + (0.2 * precision) + (0.1 * fuzzy)

## 8. `_margin_for`

This function checks how far ahead the best answer is.

Why that matters:
- if the best answer is only a tiny bit better than the second answer, the bot should be careful
- if the best answer is clearly ahead, the bot can be more confident

So this function helps the chatbot avoid guessing.

In [10]:
def _margin_for(candidates: tuple[MatchCandidate, ...]) -> float:
    if len(candidates) < 2:
        return candidates[0].score if candidates else 0.0
    return candidates[0].score - candidates[1].score

## 9. `Matcher.match`

This is a promise, not a finished function.

It says:
- every matcher in this project must have a `match` function
- that function takes a user question and the FAQ list
- that function returns either a good match or `None`

This keeps the project organized.

In [10]:
class Matcher(ABC):
    """Contract shared by the local matcher and future LLM matcher."""

    @abstractmethod
    def match(self, query: str, faqs: list[FAQ]) -> MatchResult | None:
        raise NotImplementedError

## 10. `LexicalMatcher.__init__`

This method sets up the simple backup matcher.

It stores two limits:
- `min_score`: how strong the match must be
- `min_margin`: how far ahead the best answer must be

These limits stop the backup matcher from being too bold.

In [11]:
def __init__(
    self,
    min_score: float = DEFAULT_LEXICAL_THRESHOLD,
    min_margin: float = DEFAULT_LEXICAL_MARGIN,
) -> None:
    self.min_score = min_score
    self.min_margin = min_margin

## 11. `LexicalMatcher._score`

This method scores one FAQ against one user question.

It does not do the work itself.
It simply calls the text-comparison helper on the full FAQ document.

In [12]:
def _score(self, query: str, faq: FAQ) -> float:
    return _lexical_similarity(query, build_faq_document(faq))

## 12. `LexicalMatcher.match`

This is the full backup matching flow.

It does these steps:
- stop early if there are no FAQs
- score every FAQ
- sort them from best to worst
- keep only the top few
- check the score limit
- check the gap between first and second place
- return the winner only if it looks safe enough

If not, it returns `None`.

In [13]:
def match(self, query: str, faqs: list[FAQ]) -> MatchResult | None:
    if not faqs:
        return None

    candidates = tuple(
        sorted(
            (
                MatchCandidate(faq=faq, score=self._score(query, faq))
                for faq in faqs
            ),
            key=lambda candidate: candidate.score,
            reverse=True,
        )[:3]
    )

    if not candidates:
        return None

    margin = _margin_for(candidates)
    if candidates[0].score < self.min_score or margin < self.min_margin:
        return None

    return MatchResult(
        faq=candidates[0].faq,
        score=candidates[0].score,
        margin=margin,
        matcher_name="lexical",
        candidates=candidates,
    )

## 13. `SemanticMatcher.__init__`

This method sets up the main matcher.

It stores:
- the model name
- the safety limits
- whether debug mode is on
- the logger function for printing messages
- the model itself, if one was already passed in
- saved FAQ embeddings so they can be reused
- the fallback matcher

In plain words: this is the setup box for the smarter matcher.

In [14]:
def __init__(
    self,
    model_name: str = MODEL_NAME,
    min_score: float = DEFAULT_MIN_SCORE,
    min_margin: float = DEFAULT_MIN_MARGIN,
    debug: bool = False,
    logger: Callable[[str], None] | None = None,
    model: Any | None = None,
    lexical_fallback: Matcher | None = None,
) -> None:
    self.model_name = model_name
    self.min_score = min_score
    self.min_margin = min_margin
    self.debug = debug
    self.logger = logger or (lambda _: None)
    self._model = model
    self._faq_signature: tuple[tuple[int, str, str], ...] | None = None
    self._faq_embeddings: np.ndarray | None = None
    self._fallback = lexical_fallback or LexicalMatcher()
    self._warned_about_fallback = False

## 14. `SemanticMatcher._log`

This method prints a message only when debug mode is turned on.

That means:
- normal users do not see extra noise
- the developer can still inspect what the matcher is doing

In [15]:
def _log(self, message: str) -> None:
    if self.debug:
        self.logger(message)

## 15. `SemanticMatcher._load_model`

This method loads the sentence model only when it is actually needed.

Why this is useful:
- the app starts more simply
- the model is loaded once
- later questions can reuse the same model

In [16]:
def _load_model(self) -> Any:
    if self._model is None:
        from sentence_transformers import SentenceTransformer

        self._model = SentenceTransformer(self.model_name)
    return self._model

## 16. `SemanticMatcher._signature_for`

This method builds a simple fingerprint for the FAQ list.

In simple words:
- if the FAQs are the same as last time, the fingerprint matches
- if the FAQs changed, the fingerprint changes too

That helps the chatbot know whether it can keep using saved embeddings.

In [17]:
def _signature_for(self, faqs: list[FAQ]) -> tuple[tuple[int, str, str], ...]:
    return tuple((faq.id, faq.question, faq.answer) for faq in faqs)

## 17. `SemanticMatcher._encode_documents`

This method turns the FAQ documents into number lists.

It first checks whether the model has a special document function.
If yes, it uses it.
If not, it uses the normal encode function.

Either way, it gives back the FAQ embeddings.

In [18]:
def _encode_documents(self, model: Any, documents: list[str]) -> np.ndarray:
    if hasattr(model, "encode_document"):
        return np.asarray(model.encode_document(documents, convert_to_numpy=True))
    return np.asarray(model.encode(documents, convert_to_numpy=True))

## 18. `SemanticMatcher._encode_query`

This does the same kind of job as the last function, but for the user's question.

It turns the user question into a number list that can be compared with the FAQ number lists.

In [19]:
def _encode_query(self, model: Any, query: str) -> np.ndarray:
    if hasattr(model, "encode_query"):
        return np.asarray(model.encode_query(query, convert_to_numpy=True))
    return np.asarray(model.encode(query, convert_to_numpy=True))

## 19. `SemanticMatcher._ensure_embeddings`

This method makes sure the FAQ embeddings exist.

It checks:
- are the FAQs the same as last time?
- do we already have saved embeddings?

If yes, it does nothing.
If no, it rebuilds the FAQ documents and encodes them.

This saves a lot of repeated work.

In [20]:
def _ensure_embeddings(self, faqs: list[FAQ]) -> None:
    signature = self._signature_for(faqs)
    if self._faq_signature == signature and self._faq_embeddings is not None:
        return

    model = self._load_model()
    documents = [build_faq_document(faq) for faq in faqs]
    self._faq_embeddings = self._encode_documents(model, documents)
    self._faq_signature = signature

## 20. `SemanticMatcher._rank`

This method scores every FAQ and puts the best ones at the top.

It uses two ideas together:
- the model score from the number-list comparison
- a small extra boost if the FAQ question itself looks like the user's wording

Then it keeps only the top three answers.

This is helpful because sometimes the FAQ answer text can pull the wrong item upward. The small question bonus helps the more natural match win.

In [11]:
def _rank(self, query: str, query_embedding: np.ndarray, faqs: list[FAQ]) -> tuple[MatchCandidate, ...]:
    assert self._faq_embeddings is not None
    candidates = []
    for faq, embedding in zip(faqs, self._faq_embeddings):
        semantic_score = _cosine_similarity(query_embedding, embedding)
        question_hint = _lexical_similarity(query, faq.question)
        final_score = min(1.0, semantic_score + (QUESTION_HINT_WEIGHT * question_hint))
        candidates.append(MatchCandidate(faq=faq, score=final_score))
    return tuple(sorted(candidates, key=lambda candidate: candidate.score, reverse=True)[:3])

## 21. `SemanticMatcher._should_reject_specific_query`

This method handles one very human problem.

Example:
- the user asks for the next event date
- the FAQ only says "check our noticeboard for dates"

That is related, but it does not truly answer the question.

So this method says: reject that match.

This helps the bot avoid giving a half-answer to a very specific question.

In [22]:
def _should_reject_specific_query(self, query: str, top_candidate: MatchCandidate) -> bool:
    lowered_query = _normalize_text(query)
    asks_for_specific_event_timing = (
        ("event" in lowered_query or "events" in lowered_query)
        and any(term in lowered_query for term in ("next", "date", "dates", "time", "when"))
    )
    answer_points_elsewhere = "noticeboard for dates" in _normalize_text(top_candidate.faq.answer)
    return asks_for_specific_event_timing and answer_points_elsewhere

## 22. `SemanticMatcher._emit_debug_scores`

This method prints the top candidates when debug mode is on.

This is useful for the person building the chatbot because they can see:
- which FAQs were close
- which FAQ won
- how strong the scores were

In [23]:
def _emit_debug_scores(self, candidates: tuple[MatchCandidate, ...]) -> None:
    if not self.debug or not candidates:
        return

    debug_parts = [
        f"{candidate.faq.id}:{candidate.score:.3f} {candidate.faq.question}"
        for candidate in candidates
    ]
    self._log("Debug top candidates -> " + " | ".join(debug_parts))

## 23. `SemanticMatcher._fallback_match`

This method is the rescue plan.

If the main model breaks or is missing, this method:
- prints one warning message
- tries the lexical matcher instead
- marks the result as coming from fallback mode

So the chatbot keeps working instead of crashing.

In [24]:
def _fallback_match(self, query: str, faqs: list[FAQ], exc: Exception) -> MatchResult | None:
    if not self._warned_about_fallback:
        self.logger(
            "Bot: Embedding model unavailable, falling back to lexical matching."
        )
        self._warned_about_fallback = True
    self._log(f"Debug fallback reason -> {exc!r}")

    fallback_result = self._fallback.match(query, faqs)
    if fallback_result is None:
        return None
    return replace(fallback_result, used_fallback=True)

## 24. `SemanticMatcher.match`

This is the main brain of the local chatbot.

Step by step, it does this:
- stop if there are no FAQs
- make sure FAQ embeddings are ready
- prepare the user's question
- turn the question into numbers
- rank the FAQs
- if anything goes wrong, use the fallback matcher
- show debug info if needed
- reject weak or misleading matches
- return the final best answer

This function is where safety and matching come together.

In [25]:
def match(self, query: str, faqs: list[FAQ]) -> MatchResult | None:
    if not faqs:
        return None

    try:
        self._ensure_embeddings(faqs)
        model = self._load_model()
        prepared_query = _prepare_query(query)
        query_embedding = self._encode_query(model, prepared_query)
        candidates = self._rank(query, query_embedding, faqs)
    except Exception as exc:
        return self._fallback_match(query, faqs, exc)

    if not candidates:
        return None

    self._emit_debug_scores(candidates)
    margin = _margin_for(candidates)
    top_candidate = candidates[0]

    if self._should_reject_specific_query(query, top_candidate):
        self._log("Debug rejection -> top candidate lacked the requested specific detail.")
        return None

    if top_candidate.score < self.min_score or margin < self.min_margin:
        return None

    return MatchResult(
        faq=top_candidate.faq,
        score=top_candidate.score,
        margin=margin,
        matcher_name="semantic",
        candidates=candidates,
    )

## 25. `LlmMatcher.match`

This function is not built yet.

That is on purpose.
- the main version of this project uses the local matcher
- the LLM version is only a future plan

So this method raises an error to make that clear.

In [26]:
def match(self, query: str, faqs: list[FAQ]) -> MatchResult | None:
    raise NotImplementedError(
        "LlmMatcher is intentionally not implemented for the main submission. "
        "See the README for the scaling plan."
    )

## 26. `build_matcher`

This function creates the main matcher for the chatbot.

Right now it always returns a `SemanticMatcher`.

This is useful because the rest of the chatbot does not need to know how the matcher is built.

In [27]:
def build_matcher(
    *,
    debug: bool = False,
    logger: Callable[[str], None] = print,
) -> Matcher:
    return SemanticMatcher(debug=debug, logger=logger)

## 27. `run_cli`

This function runs the full chat loop in the terminal.

What it does:
- loads the FAQs
- builds the matcher
- prints the welcome message
- keeps asking the user for questions
- stops when the user types `quit` or `exit`
- handles empty input politely
- sends the question to the matcher
- prints the answer or the fallback message

This is the function that makes the app feel like a chatbot.

In [28]:
def run_cli(
    *,
    faq_path: Path | None = None,
    matcher: Matcher | None = None,
    debug: bool = False,
    input_func: Callable[[str], str] = input,
    output_func: Callable[[str], None] = print,
) -> int:
    faqs = load_faqs(faq_path)
    active_matcher = matcher or build_matcher(debug=debug, logger=output_func)

    output_func("=" * 50)
    output_func("  Welcome to Riverside Books!")
    output_func("  Ask me anything - type 'quit' or 'exit' to leave.")
    output_func("=" * 50)
    output_func(f"  Loaded {len(faqs)} FAQs")
    output_func("=" * 50)

    while True:
        try:
            user_input = input_func("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            output_func(f"\nBot: {GOODBYE_MESSAGE}")
            return 0

        if user_input.lower() in {"quit", "exit"}:
            output_func(f"\nBot: {GOODBYE_MESSAGE}")
            return 0

        if not user_input:
            output_func("Bot: Please ask a question.")
            continue

        result = active_matcher.match(user_input, faqs)
        if result is None:
            output_func(f"Bot: {FALLBACK_MESSAGE}")
            continue

        output_func(f"Bot: {result.faq.answer}")

## 28. `main` in `src/chatbot.py`

This function is a very small wrapper.

It simply passes the work to `run_cli`.

Why keep it?
- it gives the project a clean main entry inside the chatbot module

In [29]:
def main(*, faq_path: Path | None = None, debug: bool = False) -> int:
    return run_cli(faq_path=faq_path, debug=debug)

## 29. `parse_args`

This function reads extra options from the command line.

It supports:
- `--faq-path` to choose a different FAQ file
- `--debug` to show extra matching details

So this function lets the person running the program change how it starts.

In [30]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Riverside Books FAQ chatbot")
    parser.add_argument(
        "--faq-path",
        type=Path,
        default=None,
        help="Optional path to a faqs.json file.",
    )
    parser.add_argument(
        "--debug",
        action="store_true",
        help="Print matcher diagnostics while answering questions.",
    )
    return parser.parse_args()

## 30. Final startup block

This is not a function, but it is worth understanding.

It means:
- if this file is run directly, start the program
- first read the command-line options
- then run the chatbot
- finally exit cleanly with the returned status code

Notebook note:
- in this notebook, we print a simple message instead of starting the live chat loop
- that keeps the notebook easy to run from top to bottom

In [31]:
if __name__ == "__main__":
    print("This final block starts the chatbot when main.py is run directly.")

This final block starts the chatbot when main.py is run directly.
